# Dive Into GridR's core grid resampling standalone mode

This notebook validates the *standalone preprocessing* path of
`gridr.core.grid.grid_resampling`. For each test case we display three
diagnostic figures side by side:

1. **Resampling inputs** &mdash; the source raster, the optional input mask
   (with its safe-valid window if any) and the grid overlay (nodes + node
   connectors).
2. **Standalone preprocessing outputs** &mdash; the padded array and mask
   as produced by `standalone_preprocessing`, together with the *minimal
   source read window* (`array_src_win_read`) and the *padded support
   window* (`array_src_win_marged`) required by the interpolator.
3. **Resampling outputs** &mdash; the resampled raster and its mask, with
   `nodata_out` cells rendered as transparent.

The plotting helpers and the test-input generators live in
`grid_resampling_plot_utils.py`; this notebook is kept as a thin driver
that exercises individual scenarios.


## Setting things up


In [ ]:
import sys

# Make the in-repo gridr package importable from the notebook location.
sys.path.insert(0, "/".join(["..", "python"]))

import numpy as np
from functools import partial

# Test-input generators, plotting helpers and top-level driver.
from grid_resampling_plot_utils import (
    main_plot,
    create_grid,
    _mono_point_grid_generate_input,
    _multi_points_grid_generate_input,
    PIXEL_SIZE,
    DPI,
)


## Mono point grids

A *mono point grid* is a 1x1 grid: a single resampling node placed at
`(row_target, col_target)`. This is the simplest setup to probe how the
resampler behaves on individual locations &mdash; in particular near the
source raster boundary and around masked cells.


### Non-uniform array_in

The source raster is filled with `np.arange(...)` (a ramp), so that any
interpolation error is immediately visible in the resampled value.


In [ ]:
# Partial application of the mono-point input generator with the defaults
# common to every nominal-case test below.
_mono_point_grid_generate_input__partial_nominal_case = partial(
    _mono_point_grid_generate_input,
    array_in_cst_value=None,         # ramp-filled source (np.arange)
    shape_array_in=(20, 30),
    resolution=(1, 1),
    set_array_in_mask_safe_window=False,
    array_out_mask=True,
    use_standalone=True,
    nodata_out=-9999.0,
    array_in_origin=(0., 0.),
    win=None,
    use_grid_mask=False,
    grid_mask_valid_value=1,
    check_boundaries=True,
)


def _mono_point_grid_generate_input__partial_nominal_case_build_kwargs(args):
    """Zip a positional `args` tuple with the names of the remaining
    (non-default) parameters of the partial above. Lets us write test
    cases as compact tuples."""
    return dict(zip(
        (
            "row_target",
            "col_target",
            "use_array_in_mask",
            "set_array_in_mask_invalid",
            "set_array_in_mask_safe_window",
            "interp",
            "boundary_condition",
            "trust_padding",
            "bsplines_kwargs",
        ),
        args,
    ))


In [ ]:
# Note: when boundary_condition is None, the safe_region is computed
# regardless of the safe-window setting.

# (row_target, col_target,
#  use_array_in_mask, set_array_in_mask_invalid, set_array_in_mask_safe_window,
#  interp, boundary_condition, trust_padding, bsplines_kwargs)
args = (0., 0.1, False, False, False, "bspline3", "reflect", False, {})
#args = (4.1, 4.1, False, False, False, "bspline3", "reflect", True, {})
# args = (9.001, 9.001, True, False, False, "bspline3", "reflect", False,
#         {'epsilon': 1e-2, 'mask_influence_threshold': 1})

# For bspline interpolators, default bspline kwargs need to be provided.
# 'mask_influence_threshold' to 1 deactivate the bspline prefiltering dilatation
# on mask
if "bspline" in args[5]:
    args = args[0:8] + ({'epsilon': 1e-2, 'mask_influence_threshold': 0.00001},)
    args = args[0:8] + ({'epsilon': 1e-2, 'mask_influence_threshold': 1},)

kw_001 = _mono_point_grid_generate_input__partial_nominal_case(
    **_mono_point_grid_generate_input__partial_nominal_case_build_kwargs(args)
)

resampled_array, resampled_mask = main_plot(**kw_001, pixel_size=100, dpi=150)

# Echo the case as a single tuple, ready to be pasted into a pytest parametrize
# fixture: (args, expected_array, expected_mask).
np_print_options = np.get_printoptions()
np.set_printoptions(linewidth=200, formatter={'float': '{:.6f}'.format})
print(f"( {args}, {resampled_array}, {resampled_mask} ),")
np.set_printoptions(**np_print_options)


## Multi nodes grid

A general grid is built with :func:`create_grid` from an affine
description: an origin node `(yo, xo)` and two basis vectors expressed
in source-pixel coordinates &mdash; one per grid axis. This is general
enough to express the most common test grids:

- *Identity*: ``origin_node = (1, 1)``, basis ``((2, 0), (0, 3))`` on a
  2x3 grid maps each grid node to the center of a 2x3-pixel source
  cell.
- *Shift row* / *shift col*: identical to identity but with a
  fractional shift on one axis, so resampling has to actually
  interpolate.


### Grid interpolation


In [ ]:
# Partial application of the multi-point generator with the defaults
# shared by the cases below.
_multi_point_grid_generate_input__partial_nominal_case_no_array_mask = partial(
    _multi_points_grid_generate_input,
    shape_array_in=(10, 15),
    # resolution is set per-case below.
    use_array_in_mask=False,
    array_in_mask_invalid_slice=None,
    array_in_mask_safe_window=None,
    array_out_mask=True,
    use_standalone=True,
    nodata_out=-9999.0,
    array_in_origin=(0., 0.),
    win=None,
    check_boundaries=True,
)


def _multi_point_grid_generate_input__partial_nominal_case_no_array_mask_build_kwargs(args):
    """Zip a positional `args` tuple with the names of the remaining
    (non-default) parameters of the partial above."""
    return dict(zip(
        (
            "create_grid_kwargs",
            "resolution",
            "array_in_cst_value",
            "use_grid_mask",
            "grid_mask_valid_value",
            "grid_mask_novalid_slice",
            "boundary_condition",
            "trust_padding",
            "interp",
            "bsplines_kwargs",
        ),
        args,
    ))


#### Without grid mask

Three pre-canned grid configurations are available below: identity,
row-shifted, column-shifted. Uncomment one and run.


In [ ]:
# Identity grid
# GRID_KWARGS = {"nrow": 2, "ncol": 3, "origin_pos": (0, 0), "origin_node": (1., 1.),
#                "v_row_y": 2., "v_row_x": 0., "v_col_y": 0., "v_col_x": 3.,
#                "grid_dtype": np.float64}
# Shift Row
# GRID_KWARGS = {"nrow": 2, "ncol": 3, "origin_pos": (0, 0), "origin_node": (1.5, 1.),
#                "v_row_y": 2., "v_row_x": 0., "v_col_y": 0., "v_col_x": 3.,
#                "grid_dtype": np.float64}
# Shift Col
GRID_KWARGS = {"nrow": 3, "ncol": 3, "origin_pos": (0, 0), "origin_node": (1., 1.2),
               "v_row_y": 2., "v_row_x": 0., "v_col_y": 0., "v_col_x": 3.,
               "grid_dtype": np.float64}

# (create_grid_kwargs, resolution, array_in_cst_value,
#  use_grid_mask, grid_mask_valid_value, grid_mask_novalid_slice,
#  boundary_condition, trust_padding, interp, bsplines_kwargs)
args = (GRID_KWARGS, (2, 3), None, False, 1, None, "reflect", True, "linear", {})

if "bspline" in args[8]:
    args = args[0:9] + ({'epsilon': 1e-2, 'mask_influence_threshold': 0.001},)
    args = args[0:9] + ({'epsilon': 1e-2, 'mask_influence_threshold': 1},)

kw_001 = _multi_point_grid_generate_input__partial_nominal_case_no_array_mask(
    **_multi_point_grid_generate_input__partial_nominal_case_no_array_mask_build_kwargs(args)
)

resampled_array, resampled_mask = main_plot(**kw_001, pixel_size=100, dpi=150)

np_print_options = np.get_printoptions()
np.set_printoptions(linewidth=200, formatter={'float': '{:.6f}'.format})
print(f"( {args}, {resampled_array}, {resampled_mask} ),")
np.set_printoptions(**np_print_options)


#### With grid mask

Same as above, but with a grid mask that flags a sub-region of the grid
as invalid. The resampler should propagate this invalidation to the
output mask without altering the resampled values inside the valid
zone.


In [ ]:
# Identity grid
# GRID_KWARGS = {"nrow": 2, "ncol": 3, "origin_pos": (0, 0), "origin_node": (1., 1.),
#                "v_row_y": 2., "v_row_x": 0., "v_col_y": 0., "v_col_x": 3.,
#                "grid_dtype": np.float64}
# Shift Row
# GRID_KWARGS = {"nrow": 2, "ncol": 3, "origin_pos": (0, 0), "origin_node": (1.5, 1.),
#                "v_row_y": 2., "v_row_x": 0., "v_col_y": 0., "v_col_x": 3.,
#                "grid_dtype": np.float64}
# Shift Col
GRID_KWARGS = {"nrow": 3, "ncol": 3, "origin_pos": (0, 0), "origin_node": (1., 1.5),
               "v_row_y": 2., "v_row_x": 0., "v_col_y": 0., "v_col_x": 3.,
               "grid_dtype": np.float64}

# Different shapes for the invalid slice of the grid mask:
# - full grid invalid:          (slice(None), slice(None))
# - no invalid (all valid):     None
# - one corner invalid:         (slice(0, 1), slice(0, 1))
grid_mask_novalid_slice = (slice(None), slice(None))
grid_mask_novalid_slice = None
grid_mask_novalid_slice = (slice(0, 1), slice(0, 1))

args = (GRID_KWARGS, (1, 1), None, True, 1, grid_mask_novalid_slice,
        "reflect", True, "linear", {})

if "bspline" in args[8]:
    args = args[0:9] + ({'epsilon': 1e-2, 'mask_influence_threshold': 0.001},)
    args = args[0:9] + ({'epsilon': 1e-2, 'mask_influence_threshold': 1},)

kw_001 = _multi_point_grid_generate_input__partial_nominal_case_no_array_mask(
    **_multi_point_grid_generate_input__partial_nominal_case_no_array_mask_build_kwargs(args)
)

resampled_array, resampled_mask = main_plot(**kw_001, pixel_size=100, dpi=150)

np_print_options = np.get_printoptions()
np.set_printoptions(linewidth=200, formatter={'float': '{:.6f}'.format})
print(f"( {args}, {resampled_array}, {resampled_mask} ),")
np.set_printoptions(**np_print_options)


### Safe window context

Outstanding scenarios to cover here:

- with / without safe window on a masked source,
- with / without boundary checking (nominal vs error path),
- standalone vs non-standalone on the nominal case,
- standalone vs non-standalone on an irregular grid.

The next two cells set up two starter cases for this section.


In [ ]:
# 10x10 grid on a 50x60 source, identity-like mapping shifted into the
# source by ~(19, 20). The input mask is fully valid; uncomment the
# alternative `array_in_mask_invalid_slice` / `array_in_mask_safe_window`
# lines to flip on a masked region with a safe-valid sub-window.
GRID_KWARGS = {"nrow": 10, "ncol": 10, "origin_pos": (0, 0),
               "origin_node": (19.1, 20.1),
               "v_row_y": 2., "v_row_x": 0., "v_col_y": 0., "v_col_x": 2.,
               "grid_dtype": np.float64}

kw_001 = _multi_points_grid_generate_input(
    array_in_cst_value=None,
    shape_array_in=(50, 60),
    create_grid_kwargs=GRID_KWARGS,
    resolution=(1, 1),
    use_array_in_mask=True,
    #array_in_mask_invalid_slice=None,
    array_in_mask_invalid_slice=(slice(4, 30), slice(13, 23)),
    #array_in_mask_safe_window=None,
    array_in_mask_safe_window=((30, 49), (23, 59)),
    # array_in_mask_safe_window=((0, 49), (0, 59)),
    array_out_mask=True,
    use_standalone=True,
    interp="bspline3",
    nodata_out=-9999,
    array_in_origin=(0., 0.),
    win=None,
    use_grid_mask=None,
    grid_mask_valid_value=1,
    grid_mask_novalid_slice=None,
    check_boundaries=True,
    boundary_condition="reflect",
    trust_padding=True,
    bsplines_kwargs={'epsilon': 1e-2, 'mask_influence_threshold': 0.001},
)
resampled_array, resampled_mask = main_plot(**kw_001, pixel_size=100, dpi=150)


In [ ]:
# Tiny 2x2 grid on a 5x5 source, fractional offset, cubic interpolator,
# no boundary condition. Useful to inspect what the resampler does on
# the corners when boundary_condition is None.
GRID_KWARGS = {"nrow": 2, "ncol": 2, "origin_pos": (0, 0),
               "origin_node": (0.1, 0.1),
               "v_row_y": 1., "v_row_x": 0., "v_col_y": 0., "v_col_x": 1.,
               "grid_dtype": np.float64}

kw_001 = _multi_points_grid_generate_input(
    array_in_cst_value=None,
    shape_array_in=(5, 5),
    create_grid_kwargs=GRID_KWARGS,
    resolution=(1, 1),
    use_array_in_mask=True,
    array_in_mask_invalid_slice=None,
    # array_in_mask_invalid_slice=(slice(4, 30), slice(13, 23)),
    array_in_mask_safe_window=None,
    # array_in_mask_safe_window=((30, 49), (23, 59)),
    # array_in_mask_safe_window=((0, 49), (0, 59)),
    array_out_mask=True,
    use_standalone=True,
    interp="cubic",
    nodata_out=-9999,
    array_in_origin=(0., 0.),
    win=None,
    use_grid_mask=None,
    grid_mask_valid_value=1,
    grid_mask_novalid_slice=None,
    check_boundaries=True,
    boundary_condition=None,
    trust_padding=False,
    bsplines_kwargs={'epsilon': 1e-2, 'mask_influence_threshold': 0.001},
)
resampled_array, resampled_mask = main_plot(**kw_001, pixel_size=100, dpi=150)
